In [7]:
!pip install faiss-cpu

   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB 435.7 kB/s eta 0:00:44
   ---------------------------------------- 0.1/18.9 MB 550.5 kB/s eta 0:00:35
   ---------------------------------------- 0.1/18.9 MB 658.3 kB/s eta 0:00:29
   ---------------------------------------- 0.2/18.9 MB 1.1 MB/s eta 0:00:17
    --------------------------------------- 0.3/18.9 MB 1.1 MB/s eta 0:00:17
    --------------------------------------- 0.5/18.9 MB 1.6 MB/s eta 0:00:12
   - -------------------------------------- 0.6/18.9 MB 1.9 MB/s eta 0:00:10
   - -------------------------------------- 0.7/18.9 MB 2.0 MB/s eta 0:00:09
   - -------------------------------------- 0.7/18.9 MB 2.0 MB/s eta 0:00:09
   - -------------------------------------- 0.7/18.9 MB 1.6 MB/s eta 0:00:12
   - -------------------------------------- 0.7/18.9 MB 1.6 MB/s eta 0:00:12
   -- -

In [8]:
import pandas as pd
import numpy as np
import pickle
import faiss

In [9]:
hotels = pd.read_csv("hotels.csv")
places = pd.read_csv("places.csv")
restaurants = pd.read_csv("restaurants.csv")

print("Hotels:", hotels.shape)
display(hotels.head())

print("Places:", places.shape)
display(places.head())

print("Restaurants:", restaurants.shape)
display(restaurants.head())

Hotels: (2550, 10)


,id,city,Hotel Name,Price,Review Score (/10),Hotel URL,Description,Latitude,Longitude,Distance (km)
0,1,alexandria,Tolip Hotel Alexandria,543.0,8.2,https://www.booking.com/hotel/eg/royal-tulip-a...,This 5-star hotel in Alexandria offers an outd...,31.231793,29.946027,180.59
1,2,alexandria,Rixos Montaza Alexandria,779.0,9.1,https://www.booking.com/hotel/eg/rixos-montaza...,Prime Location: Rixos Montaza Alexandria in Al...,31.288405,30.015993,180.78
2,3,alexandria,Alexandria Ap,140.0,7.5,https://www.booking.com/hotel/eg/alexandria-ap...,Comfortable Accommodations: Alexandria Ap in A...,31.240590,29.956584,180.61
3,4,alexandria,Hilton Alexandria Green Plaza,343.0,8.4,https://www.booking.com/hotel/eg/hilton-alexan...,"On site of Green Plaza Mall, this Hilton is on...",31.206543,29.965575,177.28
4,5,alexandria,Palestine Montaza Alexandria,700.0,8.4,https://www.booking.com/hotel/eg/helnan-palest...,Set on 350 acres of lush gardens overlooking a...,31.287894,30.017722,180.63


Places: (391, 10)


,id,City,Title,Rating,Reviews,Description,Tips,Address,Timings,Ticket Price
0,1,cairo,Pyramids Of Giza,4.0,7224.0,The most famous of the Seven Wonders of the An...,Avoid visiting Pyramids of Giza at night time....,"Al Ahram, Al Haram, Giza, Egypt, Cairo",07:00 am - 07:00 pm,60.0
1,2,cairo,Keops Pyramid,NaN,NaN,Take a historic walk through the ancient icons...,Guided tours are available and recommended.\nD...,"Al Haram, Nazlet El-Semman, Al Haram, Giza Gov...",08:00 am - 05:00 pm,NaN
2,3,cairo,Sphinx,4.0,6728.0,"The Terrifying One, as the Great Sphinx is kno...",Confirm timings before visiting the Sphinx.\nP...,"Al Ahram, Giza, Egypt, Cairo",08:00 am - 05:00 pm,20.0
3,4,cairo,Islamic Cairo,4.0,2752.0,Islamic Cairo is a historic area that was crea...,Guided tours are available.,"Al Gamaleyah, Qesm Gamaleyah, Cairo Governorat...",24-hrs,NaN
4,5,cairo,The Egyptian Museum,4.0,5544.0,This Egyptian museum is located at Tahrir squa...,Photography is strictly prohibited.\nCameras w...,"Tahrir Square, Meret Basha, Ismailia, Qasr an ...",09:00 am - 07:00 pm,75.0


Restaurants: (1225, 11)


,id,city,Restaurant,Location,Cuisines,URL,Total_Items,Prices_List,Min_Price,Max_Price,Avg_Price
0,1,alexandria,Teus,"inSemouha - Semouha Sporting Club,Egypt","Sandwiches, Burgers, Fast Food",https://www.talabat.com/egypt/restaurant/66109...,183,"[158.0, 137.0, 85.0, 165.0, 56.0, 28.0, 137.0,...",11.0,199.0,89.818713
1,2,alexandria,D's Patisserie,"inSemouha,Egypt","Bakery & Pastry, Desserts, Arabic sweets",https://www.talabat.com/egypt/restaurant/73112...,16,"[65.0, 200.0, 110.0, 75.0, 250.0, 85.0, 180.0,...",65.0,250.0,135.625000
2,3,alexandria,T-lab,"inKafr Abdo,Egypt","Cafe, Beverages",https://www.talabat.com/egypt/restaurant/78301...,50,"[230.0, 230.0, 230.0, 20.0]",20.0,230.0,177.500000
3,4,alexandria,The Canteen,"inKafr Abdo,Egypt","Healthy , Chicken, Salad, Sandwiches",https://www.talabat.com/egypt/restaurant/51731...,7,"[179.0, 249.0, 249.0, 179.0, 239.0, 249.0, 249.0]",179.0,249.0,227.571429
4,5,alexandria,Ma3booj,"inIbrahimia,Egypt","Arabic, Saudi, Kuwaiti",https://www.talabat.com/egypt/restaurant/76058...,48,"[50.0, 60.0, 80.0, 99.0, 149.0, 149.0, 129.0, ...",15.0,500.0,77.369565


In [10]:
embeddings = np.load("embeddings.npy")
print("Embeddings shape:", embeddings.shape)

Embeddings shape: (4166, 384)


In [11]:
with open("travel_metadata.pkl", "rb") as f:
    metadata = pickle.load(f)

print(type(metadata))
print(metadata[:3])  # preview first 3 entries

<class 'pandas.core.frame.DataFrame'>
    type                      name        city  price  rating  \
0  hotel    Tolip Hotel Alexandria  alexandria  543.0     8.2   
1  hotel  Rixos Montaza Alexandria  alexandria  779.0     9.1   
2  hotel             Alexandria Ap  alexandria  140.0     7.5   

                                                text  
0  Hotel Tolip Hotel Alexandria in alexandria wit...  
1  Hotel Rixos Montaza Alexandria in alexandria w...  
2  Hotel Alexandria Ap in alexandria with price 1...  


In [12]:
index = faiss.read_index("travel_vector_db.index")
print("Total vectors:", index.ntotal)
print("Dimensions:", index.d)

Total vectors: 4166
Dimensions: 384


In [13]:
with open("db_queries.sql") as f:
    print(f.read())

with open("insert_data.sql") as f:
    print(f.read())

CREATE DATABASE eg_trip_db;
USE eg_trip_db;


-- =============================================
-- 1. dim_hotels
-- =============================================
CREATE TABLE dim_hotels (
    hotel_id      INT            IDENTITY(1,1)  PRIMARY KEY,
    city          NVARCHAR(100)  NOT NULL,
    hotel_name    NVARCHAR(255)  NOT NULL,
    price         FLOAT          NULL,
    review_score  DECIMAL(3,1)   NULL,
    hotel_url     NVARCHAR(MAX)  NULL,
    description   NVARCHAR(MAX)  NULL,
    latitude      FLOAT          NULL,
    longitude     FLOAT          NULL,
    distance_km   DECIMAL(6,2)   NULL,
);

-- =============================================
-- 2. dim_places
-- =============================================
CREATE TABLE dim_places (
    place_id      INT            IDENTITY(1,1)  PRIMARY KEY,
    city          NVARCHAR(100)  NOT NULL,
    title         NVARCHAR(255)  NOT NULL,
    rating        DECIMAL(3,1)   NULL,
    reviews       FLOAT          NULL,
    description   NVARC